# Module 2: Data Cleaning & Transformation
Cleans `hospital_raw_data.csv` and produces `hospital_cleaned.csv` (Tableau-ready).

In [ ]:
import pandas as pd

df = pd.read_csv("hospital_raw_data.csv", parse_dates=["Date of Admission", "Discharge Date"])
print(df.shape)
df.head()


## 1. Remove duplicate records

In [ ]:
before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset="Patient_ID")
print(f"Removed {before - len(df)} duplicate rows")


## 2. Handle missing patient data

In [ ]:
missing_pct = df.isna().mean() * 100
print(missing_pct[missing_pct > 0])

# Fill categorical gaps with 'Unknown', numeric gaps with median
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown")

for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

print("Remaining missing %:", df.isna().mean().mean() * 100)


## 3. Standardize department names

In [ ]:
df["Department"] = df["Department"].str.strip().str.title()
print(df["Department"].unique())


## 4. Normalize healthcare indicators

In [ ]:
# Fix invalid negative billing amounts
invalid_billing = (df["Billing Amount"] < 0).sum()
print(f"Invalid negative billing rows: {invalid_billing}")
df["Billing Amount"] = df["Billing Amount"].abs()

# Recompute length of stay from actual dates for consistency
df["Length_of_stay"] = (df["Discharge Date"] - df["Date of Admission"]).dt.days
df = df[df["Length_of_stay"] >= 0]

# Standardize text casing on key categorical fields
for col in ["Admission Type", "Readmission_Flag", "Occupancy_Status", "Hospital_Region", "Gender"]:
    df[col] = df[col].astype(str).str.strip().str.title()


## 5. Create Tableau-ready dataset

In [ ]:
final_missing_pct = df.isna().mean().mean() * 100
print(f"Final missing value %: {final_missing_pct:.2f}%")
assert final_missing_pct < 2, "Missing values exceed 2% threshold"

df.to_csv("hospital_cleaned.csv", index=False)
print(f"Saved cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns")
